# Темы 9-11: Классификация (ОТВЕТЫ)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve

df = pd.merge(pd.read_csv('data/transactions.csv'), pd.read_csv('data/customers.csv'), on='customer_id')
df = pd.merge(df, pd.read_csv('data/products.csv'), on='product_id')
df['total_sum'] = df['price'] * df['quantity']
df['age'] = df['age'].fillna(df['age'].median())

customer_spend = df.groupby('customer_id')['total_sum'].sum().reset_index()
threshold = customer_spend['total_sum'].quantile(0.25)
customer_spend['churn'] = (customer_spend['total_sum'] < threshold).astype(int)

cust = pd.read_csv('data/customers.csv')
cust['age'] = cust['age'].fillna(cust['age'].median())
cust = pd.merge(cust, customer_spend[['customer_id','churn']], on='customer_id')
cust['gender_enc'] = cust['gender'].map({'M':1,'F':0})

### Задание 1

In [ ]:
print(f"Отток: {cust['churn'].mean()*100:.1f}%")
sns.countplot(x='churn', data=cust)
plt.title('Баланс классов')
plt.show()

### Задание 2

In [ ]:
X = cust[['age','gender_enc']]
y = cust['churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Задание 3

In [ ]:
lr = LogisticRegression()
lr.fit(X_train, y_train)
preds = lr.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, preds):.4f}')

### Задание 4

In [ ]:
print(classification_report(y_test, preds, zero_division=0))

### Задание 5

In [ ]:
cm = confusion_matrix(y_test, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

### Задание 6

In [ ]:
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)
dt_preds = dt.predict(X_test)
print(f'DT Accuracy: {accuracy_score(y_test, dt_preds):.4f}')

### Задание 7

In [ ]:
proba = lr.predict_proba(X_test)[:,1]
fpr, tpr, _ = roc_curve(y_test, proba)
auc = roc_auc_score(y_test, proba)
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label=f'AUC = {auc:.4f}')
plt.plot([0,1],[0,1],'--', color='gray')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('ROC Curve')
plt.legend()
plt.show()

### Задание 8

In [ ]:
tx_stats = df.groupby('customer_id').agg(
    transaction_count=('transaction_id','count'),
    avg_spend=('total_sum','mean')
).reset_index()
cust2 = pd.merge(cust, tx_stats, on='customer_id')
X2 = cust2[['age','gender_enc','transaction_count','avg_spend']]
y2 = cust2['churn']
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.2, random_state=42)
lr2 = LogisticRegression(max_iter=1000).fit(X2_tr, y2_tr)
print(f'AUC: {roc_auc_score(y2_te, lr2.predict_proba(X2_te)[:,1]):.4f}')

### Задание 9

In [ ]:
proba2 = lr2.predict_proba(X2_te)[:,1]
preds_03 = (proba2 >= 0.3).astype(int)
print('Порог 0.3:')
print(classification_report(y2_te, preds_03, zero_division=0))

### Задание 10

In [ ]:
importance = pd.DataFrame({'feature': X2.columns, 'importance': dt.fit(X2_tr, y2_tr).feature_importances_})
print(importance.sort_values('importance', ascending=False))

### Задание 11

In [ ]:
cm = confusion_matrix(y2_te, lr2.predict(X2_te))
fp = cm[0,1]  # False Positive
fn = cm[1,0]  # False Negative
cost = fp * 500 + fn * 5000
print(f'FP: {fp}, FN: {fn}')
print(f'Стоимость ошибок: {cost} руб.')

### Задание 12

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(LogisticRegression(max_iter=1000), X2, y2, cv=5, scoring='roc_auc')
print(f'Средний AUC: {scores.mean():.4f} ± {scores.std():.4f}')